In [33]:
!pip install langchain faiss-cpu groq langchain-huggingface tiktoken

In [34]:
!pip install langchain_community

In [35]:
from langchain_community.document_loaders import PyPDFLoader

In [36]:
loader = PyPDFLoader("/content/Chronicles of Technology.pdf")

In [37]:
!pip install pypdf

In [38]:
loader_obj = loader.load()

In [39]:
print(loader_obj[0].page_content)

Chronicles of Technology - Chapter 1
Artificial Intelligence: Technology connected people across different continents. The world continued
to evolve with remarkable innovations. Creative minds shaped the future of civilization. Technology
connected people across different continents. Communities adapted to rapid changes in society.
Ocean Mysteries: Exploration opened doors to endless possibilities. Scientists worked tirelessly to
improve human life. Technology connected people across different continents. Scientists worked
tirelessly to improve human life. Education became more accessible through digital systems.
Artificial Intelligence: The journey toward knowledge never truly ends. Exploration opened doors to
endless possibilities. Communities adapted to rapid changes in society. New discoveries inspired
generations of learners. Communities adapted to rapid changes in society.
Cyber Security: Scientists worked tirelessly to improve human life. Technology connected people
across diffe

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 5000, chunk_overlap = 1000)
chunks = splitter.split_documents(loader_obj)

In [41]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-06T18:09:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-06T18:09:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/Chronicles of Technology.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1'}, page_content='Chronicles of Technology - Chapter 1\nArtificial Intelligence: Technology connected people across different continents. The world continued\nto evolve with remarkable innovations. Creative minds shaped the future of civilization. Technology\nconnected people across different continents. Communities adapted to rapid changes in society.\nOcean Mysteries: Exploration opened doors to endless possibilities. Scientists worked tirelessly to\nimprove human life. Technology connected people across different continents. Scientists worked\ntirelessly to improve human life. Education became 

In [42]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [44]:
from langchain_community.vectorstores import FAISS

In [46]:
vector_stores = FAISS.from_documents(
    chunks, embeddings
)

In [66]:
retriever = vector_stores.as_retriever(search_type ="mmr", search_kwargs = {"k":3 })

In [48]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.5.0
    Uninstalling groq-1.5.0:
      Successfully uninstalled groq-1.5.0


In [51]:
import os
os.environ["GROQ_API_KEY"]="YOUR API KEY"

In [52]:
from langchain_groq import ChatGroq
model_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature= 0.5)

In [53]:
# print(model_llm.invoke("hi"))

content='Hi! How are you today? Is there something I can help you with or would you like to chat?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 11, 'total_tokens': 34, 'completion_time': 0.052272681, 'completion_tokens_details': None, 'prompt_time': 0.000150779, 'prompt_tokens_details': None, 'queue_time': 0.02331042, 'total_time': 0.05242346}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_fa001c7fdb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f08b6-8ad0-7c02-867f-8b424c7fc841-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34}


In [54]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template = """
    you are smart book author on technology
    answer me around the context only
    if you dont know the reply then say i dont know

    {context}

    question : {question}
    """,
    input_variables = ['context', 'question'], validate_template= True
)

In [67]:
question = "what is machine learning"

context = retriever.invoke(question)

In [68]:
report = prompt.invoke({"context": context , "question": question})

In [69]:
report

StringPromptValue(text="\n    you are smart book author on technology \n    answer me around the context only\n    if you dont know the reply then say i dont know\n\n    [Document(id='9b99b033-acd5-440d-b463-7112fca91aee', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-06T18:09:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-06T18:09:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/Chronicles of Technology.pdf', 'total_pages': 40, 'page': 33, 'page_label': '34'}, page_content='Machine Learning: Scientists worked tirelessly to improve human life. Scientists worked tirelessly to\\nimprove human life. Researchers discovered new ways to solve complex problems. Researchers\\ndiscovered new ways to solve complex problems. Researchers discovered new ways to solve complex\\nproblems.\\nCyber Security: New discoveries inspired generations of learners

In [70]:
final_answer = model_llm.invoke(report)

In [72]:
final_answer.content

"Based on the provided document, Machine Learning is mentioned as a field where:\n\n* Researchers discovered new ways to solve complex problems.\n* Education became more accessible through digital systems.\n* Technology connected people across different continents.\n* Exploration opened doors to endless possibilities.\n* Communities adapted to rapid changes in society.\n\nIt appears that Machine Learning is a field that involves using technology to improve human life, solve complex problems, and make education more accessible.\n\nIn simple terms, **Machine Learning is a type of technology that enables computers to learn and improve their performance on a task without being explicitly programmed for that task.** \n\n(I derived this general definition from my understanding of the context, although it's not explicitly stated in the provided document.)"